# Historical experiment

Preserved for provenance only. This notebook has known methodological or portability issues; use the corrected notebook linked from the root README. Outputs have been cleared.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from tensorflow import keras 

In [ ]:
dataSet = pd.read_csv("/kaggle/input/weather-dataset/weatherHistory.csv")
dataSet.head()

In [ ]:
dataSet.corr()

In [ ]:
dataSet.drop(['Formatted Date'],axis = 1 , inplace = True)
dataSet.drop(['Daily Summary'],axis = 1 , inplace = True)
dataSet.drop(['Summary'],axis = 1 , inplace = True)
dataSet.drop(['Visibility (km)'],axis = 1 , inplace = True)
dataSet.drop(['Wind Bearing (degrees)'],axis = 1 , inplace = True)
dataSet.drop(['Loud Cover'],axis = 1 , inplace = True)

dataSet.replace(to_replace= np.nan, value=0, inplace=True)
dataSet.replace(to_replace="rain", value=1, inplace=True)
dataSet.replace(to_replace="snow", value=2, inplace=True)
dataSet.head()

In [ ]:
dataSet.describe()

In [ ]:
c = dataSet.corr().nlargest(6,'Apparent Temperature (C)')['Apparent Temperature (C)'].index
plt.figure()
seaborn.heatmap(dataSet[c].corr(),annot=True)

In [ ]:
target  = dataSet['Apparent Temperature (C)']
print(target[1])

In [ ]:
dataSet.drop(['Apparent Temperature (C)'],axis = 1 , inplace = True)

In [ ]:
scaler = MinMaxScaler()
dataSet[['Precip Type','Humidity','Wind Speed (km/h)','Pressure (millibars)']] = scaler.fit_transform(dataSet[['Precip Type','Humidity','Wind Speed (km/h)','Pressure (millibars)']])
dataSet.head()

In [ ]:
features = []
for index, row in dataSet.iterrows():
    t = []
    for i in row:
        t.append(i)
    features.append(t)

In [ ]:
print(features[0])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.1)

In [ ]:
X_train = np.array(X_train)
X_test = np.array(X_test)
y_train = np.array(y_train)
y_test = np.array(y_test)

In [ ]:
X_train = np.reshape(X_train, (X_train.shape[0], 1, X_train.shape[1]))
X_test = np.reshape(X_test, (X_test.shape[0], 1, X_test.shape[1]))

In [ ]:
X_val = X_train[:15000]
y_val = y_train[:15000]
X_train = X_train[15000:]
y_train = y_train[15000:]

In [ ]:
print(X_train.shape)
print(X_test.shape)

In [ ]:
temp_model = keras.Sequential()
temp_model.add(keras.layers.Bidirectional(keras.layers.LSTM(64,input_shape=X_train.shape)))
temp_model.add(keras.layers.Dense(256, activation="relu"))
temp_model.add(keras.layers.Dense(256, activation="relu"))
temp_model.add(keras.layers.Dense(128, activation="relu"))
temp_model.add(keras.layers.Dense(64, activation="relu"))
temp_model.add(keras.layers.Dense(32, activation="relu"))
temp_model.add(keras.layers.Dense(16, activation="relu"))
temp_model.add(keras.layers.Dense(1, activation="linear"))

In [ ]:
earlystopping = keras.callbacks.EarlyStopping(monitor="val_loss",mode ="min",patience = 10)

In [ ]:
temp_model.compile(loss='mean_absolute_error', optimizer='adam', 
                   metrics=['mean_absolute_error'])
fitModel = temp_model.fit(X_train, y_train, epochs=95, batch_size=450, 
                validation_data=(X_val, y_val), verbose=1, callbacks =[earlystopping])
results = temp_model.evaluate(X_test, y_test)

In [ ]:
for i in range(122,130):
    print("the model Predicted :",temp_model.predict(X_test[i].reshape((1,1,5))))
    print("the actual value : ",y_test[i])

In [ ]:
predicted_values = []
for i in X_test:
    predicted_values.append(temp_model.predict(i.reshape((1,1,5))))

In [ ]:
seaborn.distplot(a=y_test, hist=False)
seaborn.distplot(a=predicted_values, hist=False)
plt.show()

In [ ]:
model_json = temp_model.to_json()
with open("Mytemp_model.json", "w") as json_file:
        json_file.write(model_json)
temp_model.save_weights("Mytemp_model.h5")
print("Saved model to disk")